In [ ]:
## Check to make sure theres all files

from pathlib import Path

data_folder = Path("data")
trends_folder = data_folder / "trends"

files = sorted(trends_folder.glob("*.csv"))

# Analyze Daily Google Search Interest

Combine quarterly Google Trends downloads for five stock tickers
covering 2020–2022

Check daily coverage and identify each ticker's highest search-interest
value and earliest matching date each month

Values retain their original download scales
Cross-quarter calibration is performed in the warehouse ETL notebook

### Part 3

### Read and combine files

In [ ]:
from pathlib import Path
from io import StringIO
import pandas as pd


tickers = ["AAPL", "AMZN", "GOOGL", "MSFT", "TSLA"]

files = sorted(trends_folder.glob("*.csv"))

print("CSV files found:", len(files))

if len(files) != 20:
    raise ValueError("Expected 20 CSV files in data/trends")

all_data = []

for file in files:
    # Find the header below Google's introductory lines.
    lines = file.read_text(encoding="utf-8-sig").splitlines()

    header_row = next(
        (
            i for i, line in enumerate(lines)
            if line.split(",")[0].strip().strip('"') == "Day"
        ),
        None
    )

    if header_row is None:
        raise ValueError(
            f"No daily 'Day' column found in {file.name}"
        )

    # Read the CSV starting at the actual header.
    data = pd.read_csv(
        StringIO("\n".join(lines[header_row:]))
    )

    data["Day"] = pd.to_datetime(
        data["Day"],
        errors="raise"
    )

    recognized_tickers = []

    for column in data.columns:
        ticker = column.split(":")[0].strip()

        if ticker in tickers:
            recognized_tickers.append(ticker)

            values = data[column].astype(str).str.strip()

            # Use 0.5 for Google's "<1" values.
            values = values.replace("<1", "0.5")
            values = pd.to_numeric(values, errors="raise")

            ticker_data = pd.DataFrame({
                "Date": data["Day"],
                "Ticker": ticker,
                "Interest": values,
                "Source_File": file.name
            })

            all_data.append(ticker_data)

    if not recognized_tickers:
        raise ValueError(
            f"No expected ticker columns found in {file.name}"
        )

    # Print one summary for each file.
    print(
        file.name,
        "| Rows:", len(data),
        "| Tickers:", ", ".join(recognized_tickers),
        "| Dates:", data["Day"].min().date(),
        "to", data["Day"].max().date()
    )

# Combine the data from all files.
trends_daily = pd.concat(all_data, ignore_index=True)

# Keep only the assignment period.
trends_daily = trends_daily[
    (trends_daily["Date"] >= "2020-01-01") &
    (trends_daily["Date"] < "2023-01-01")
].copy()

trends_daily = trends_daily.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

print("\nFiles successfully read:", len(files))
print("Total combined ticker-day rows:", len(trends_daily))

print("\nRows per ticker:")
print(trends_daily.groupby("Ticker").size())

print("\nSample combined data:")
print(trends_daily.head().to_string(index=False))


#Read tickets to make sure files are read correctly as i downloaded with some inconsistents

## Coverage check

In [ ]:
# Check for duplicate ticker-date combinations.
duplicates = trends_daily.duplicated(
    subset=["Ticker", "Date"],
    keep=False
)

if duplicates.any():
    print(trends_daily.loc[duplicates].to_string(index=False))
    raise ValueError("Duplicate ticker-date records found")

# Each ticker should have all 1,096 calendar days.
expected_dates = pd.date_range("2020-01-01", "2022-12-31")

for ticker in tickers:
    ticker_data = trends_daily[
        trends_daily["Ticker"] == ticker
    ]

    missing_dates = expected_dates.difference(
        pd.DatetimeIndex(ticker_data["Date"])
    )

    missing_values = ticker_data["Interest"].isna().sum()

    print(
        ticker,
        "| Days:", len(ticker_data),
        "| Missing dates:", len(missing_dates),
        "| Missing values:", missing_values
    )

    if len(missing_dates) > 0 or missing_values > 0:
        raise ValueError(f"Incomplete data for {ticker}")

if not trends_daily["Interest"].between(0, 100).all():
    raise ValueError("Interest values must be between 0 and 100")



### Find each ticker’s highest interest and date for every month

In [ ]:
trends_daily["Year"] = trends_daily["Date"].dt.year
trends_daily["Month"] = trends_daily["Date"].dt.month

# Create the 36 months in the assignment period.
monthly_results = pd.DataFrame(
    [
        (year, month)
        for year in range(2020, 2023)
        for month in range(1, 13)
    ],
    columns=["Year", "Month"]
)

for ticker in tickers:
    ticker_data = trends_daily[
        trends_daily["Ticker"] == ticker
    ].sort_values("Date")

    # If dates tie for the maximum, choose the earliest date.
    max_indices = ticker_data.groupby(
        ["Year", "Month"]
    )["Interest"].idxmax()

    monthly_max = ticker_data.loc[
        max_indices,
        ["Year", "Month", "Interest", "Date"]
    ].copy()

    monthly_max = monthly_max.rename(columns={
        "Interest": f"{ticker}_Max_Interest",
        "Date": f"{ticker}_Max_Interest_Date"
    })

    monthly_results = monthly_results.merge(
        monthly_max,
        on=["Year", "Month"],
        how="left",
        validate="one_to_one"
    )

print("Table shape:", monthly_results.shape)

# Show the entire table in the notebook.
display(monthly_results)

## Save both tables

In [ ]:
monthly_results.to_csv(
    data_folder / "trends_monthly_max.csv",
    index=False,
    date_format="%Y-%m-%d"
)

trends_daily.to_csv(
    data_folder / "trends_daily.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("Saved monthly maxima and combined daily data")